<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>14. Tablas dinámicas y reestructuración</font>  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>



---

## <font color="#5E5E5E">1. Creación de tablas dinámicas</font>

### <font color="#5E5E5E">  `DataFrame.pivot()`  </font>

El método `DataFrame.pivot()` cambia la disposición de los datos sin resumirlos: toma los valores de una o varias columnas para formar el índice de filas, las columnas del resultado y sus celdas. No calcula una agregación. Para cada pareja de etiquetas de fila y columna debe haber, como máximo, una observación.

In [2]:
import pandas as pd

# Se crea un DataFrame ad-hoc
ventas = [('Albacete', 22.5, 'Queso', 25),
         ('Villarrobledo', 21, 'Queso', 23.5),
         ('Albacete', 50, 'Jamón', 60),
         ('La Roda', 20.0, 'Aceite', 25),
         ('Villarrobledo', 47, 'Jamón', 55),
         ('La Roda', 50.0, 'Jamón', 55)]

columnas = ['Localidad', 'Precio', 'Producto', 'P.V.P']
indice =['Tienda 1', 'Tienda 2', 'Tienda 1', 'Tienda 3', 'Tienda 2', 'Tienda 3']

df_compras = pd.DataFrame(ventas, columns=columnas, index=indice)
df_compras

,Localidad,Precio,Producto,P.V.P
Tienda 1,Albacete,22.5,Queso,25.0
Tienda 2,Villarrobledo,21.0,Queso,23.5
Tienda 1,Albacete,50.0,Jamón,60.0
Tienda 3,La Roda,20.0,Aceite,25.0
Tienda 2,Villarrobledo,47.0,Jamón,55.0
Tienda 3,La Roda,50.0,Jamón,55.0


In [3]:
df_compras.pivot(index='Producto', columns='Localidad', values='Precio')

Localidad,Albacete,La Roda,Villarrobledo
Producto,,,
Aceite,NaN,20.0,NaN
Jamón,50.0,50.0,47.0
Queso,22.5,NaN,21.0


Es posible utilizar varios valores. En ese caso, la tabla creada utiliza un índice de varios niveles para las columnas. Por otra parte, si no se especifican valores, se utilizan todas las columnas. Los elementos que no existen se fijan a `NaN`.

In [4]:
#df_compras.pivot(index='Producto', columns='Localidad')       # Equivalentes
df_compras.pivot(index='Producto', columns='Localidad', values=['Precio','P.V.P'])  

Precio                          P.V.P                      
Localidad Albacete La Roda Villarrobledo Albacete La Roda Villarrobledo
Producto                                                               
Aceite         NaN    20.0           NaN      NaN    25.0           NaN
Jamón         50.0    50.0          47.0     60.0    55.0          55.0
Queso         22.5     NaN          21.0     25.0     NaN          23.5

La función `DataFrame.pivot` asume que cada par de valores *(índice, columna)* aparece **solo una vez** en la tabla original. En caso contrario, devuelve un error. 

In [5]:
ventas = [('Albacete', 22.5, 'Queso', 25),
         ('Villarrobledo', 21, 'Queso', 23.5),
         ('Albacete', 50, 'Jamón', 60),
         ('La Roda', 20.0, 'Aceite', 25),
         ('Villarrobledo', 47, 'Jamón', 55),
         ('La Roda', 50.0, 'Jamón', 55),
         ('Villarrobledo', 23, 'Queso', 22.5)] # Esta es la línea añadida

columnas = ['Localidad', 'Precio', 'Producto', 'P.V.P']
indice =['Tienda 1', 'Tienda 2', 'Tienda 1', 'Tienda 3', 'Tienda 2', 'Tienda 3', 'Tienda 4']

df_compras = pd.DataFrame.from_records(ventas, columns=columnas, index=indice)
df_compras

,Localidad,Precio,Producto,P.V.P
Tienda 1,Albacete,22.5,Queso,25.0
Tienda 2,Villarrobledo,21.0,Queso,23.5
Tienda 1,Albacete,50.0,Jamón,60.0
Tienda 3,La Roda,20.0,Aceite,25.0
Tienda 2,Villarrobledo,47.0,Jamón,55.0
Tienda 3,La Roda,50.0,Jamón,55.0
Tienda 4,Villarrobledo,23.0,Queso,22.5


In [ ]:
# Devolvería error porque (Queso, Villarrobledo) aparece dos veces.
#df_compras.pivot(index='Producto', columns='Localidad', values='Precio') 


### <font color="#5E5E5E">  `DataFrame.pivot_table()`  </font>

En su versión más sencilla, una tabla dinámica es una tabla que se construye a partir de tres columnas de otra dada, de modo que una de las columnas actúa como índice, **los valores de otra** como columnas, y otra como valores.  El método `DataFrame.pivot()` permite construir estas tablas.  Es parecido al anterior, pero utiliza una **función de agregación** para agregar datos correspondientes al mismo par fila/columna (situación anterior). Toma principalmente cuatro parámetros [**(+ inf )**](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.pivot_table.html):

* `index`/`columns`. La columna o columnas cuyos valores serán utilizados como índices/columnas en la nueva tabla. 
* `values`. Los valores de interés, sobre los que se lleva a cabo la agregación.
* `aggfunc`. La función o funciones de agregación.

Para ilustrar el uso de esta función se almacena el conjunto de datos `Titanic.csv` en un *DataFrame*, y se crea una columna con una etiqueta relativa a la franja de edad, que divide a los pasajeros en tres según sean menores de 18 (joven), mayores de 65 (anciano), o el resto (adulto).

In [7]:
df_titanic = pd.read_csv('data/titanic_prep.csv', index_col=0)
df_titanic.head()

,Sobrevive,Clase,Nombre,Sexo,Edad,Esposa/Hermanos,Padres/Hijos,Ticket,Tarifa,Número de Cabina,Puerto
Pasajero,,,,,,,,,,,
1,No,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,Yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,Yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,Yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,No,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
def franja_edad(edad):
    if edad<18: return 'joven'
    if edad>65: return 'anciano'
    return 'adulto'

df_titanic['FranjaEdad']=df_titanic['Edad'].map(franja_edad)
df_titanic['Sobrevive'] = df_titanic['Sobrevive'].map({'Yes':1,'No':0})
df_titanic.head()

,Sobrevive,Clase,Nombre,Sexo,Edad,Esposa/Hermanos,Padres/Hijos,Ticket,Tarifa,Número de Cabina,Puerto,FranjaEdad
Pasajero,,,,,,,,,,,,
1,0,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,adulto
2,1,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,adulto
3,1,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,adulto
4,1,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,adulto
5,0,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,adulto


La siguiente función muestra la tasa de supervivencia en función de los campos *Sexo* y *Clase*.

In [9]:
df_titanic.pivot_table(index='Clase', columns='Sexo', values='Sobrevive', aggfunc='mean')

Sexo,female,male
Clase,,
first,0.968085,0.368852
second,0.921053,0.157407
third,0.500000,0.135447


En este otro ejemplo, se utilizan como columnas tanto `Sexo` como `FranjaEdad` para contar los supervivientes (`aggfunc='sum'`). Además, se sustituyen los valores que no están presentes por `--`, y se añaden márgenes con el total.

In [10]:
df_titanic.pivot_table(index='Clase', columns=['Sexo', 'FranjaEdad'], values=['Sobrevive'], 
                       aggfunc='sum', fill_value='--', margins=True, margins_name='Total')

Sobrevive                                 
Sexo          female         male               Total
FranjaEdad    adulto joven adulto anciano joven      
Clase                                                
first             84     7     40       1     4   136
second            58    12      8       0     9    87
third             53    19     37       0    10   119
Total            195    38     85       1    23   342

También se pueden especificar varias columnas como valores, o varias funciones de agregación.

### <font color="#5E5E5E">  `pandas.crosstab()`  </font>

La función `pandas.crosstab()` es parecida a la anterior. En lugar de un `DataFrame` toma como argumentos las secuencias de índices, columnas y valores, por separado. Además, por defecto usa `aggfunc=len` (frecuencias) y sustituye los valores perdidos por 0.

In [11]:
sexo = df_titanic['Sexo'].values[:10]
edad = df_titanic['FranjaEdad'].values[:10]

pd.crosstab(sexo, edad, rownames=['Sexo'], colnames=['Edad'], margins=True)
#pd.crosstab(sexo, edad, rownames=['Sexo'], colnames=['Edad'], margins=True, normalize=True)
#pd.crosstab(df_titanic['Sexo'], df_titanic['FranjaEdad'], rownames=['Sexo'], colnames=['Edad'], margins=True)

Edad,adulto,joven,All
Sexo,,,
female,4,1,5
male,4,1,5
All,8,2,10


---

## <font color="#5E5E5E">2. Reestructuración de tablas</font>

### <font color="#5E5E5E">  `DataFrame.stack()`  </font>

Es posible llevar a cabo el proceso inverso al descrito anteriormente. Ese proceso se denomina **apilar** y consiste en utilizar el producto cartesiano del índice y las columnas como índice en las filas. El resultado es un `DataFrame` en el que cada fila representa una observación/variable/valor, es decir, en formato *wide*. Para ello, se utiliza la función `DataFrame.stack()`. El resultado es un `DataFrame` indexado mediante un índice de dos niveles.

In [12]:
df_precios = pd.DataFrame({'Casa':[10,20,30], 'Coche':[40,50,60], 'Transporte':[1,2,3]},
                         index = ['Madrid', 'Barcelona', 'Paris'])
df_precios

,Casa,Coche,Transporte
Madrid,10,40,1
Barcelona,20,50,2
Paris,30,60,3


In [13]:
df_precios_stack = df_precios.stack()
df_precios_stack 

Madrid     Casa          10
           Coche         40
           Transporte     1
Barcelona  Casa          20
           Coche         50
           Transporte     2
Paris      Casa          30
           Coche         60
           Transporte     3
dtype: int64


### <font color="#5E5E5E">  `DataFrame.unstack()`  </font>

El método `DataFrame.unstack()` es inversa al anterior. Utiliza el nivel inferior del índice de filas como índice de columnas. Es parecido a `DataFrame.pivot()` pero trabaja con los indices.

In [14]:
df_precios_stack.unstack()

,Casa,Coche,Transporte
Madrid,10,40,1
Barcelona,20,50,2
Paris,30,60,3


### <font color="#5E5E5E">  `DataFrame.melt()`  </font>

Otra posibilidad es `DataFrame.melt()`. Este método es parecido a `DataFrame.stack()`, pero es más flexible. Apila los nombres de varias columnas como valores de una nueva columna (en lugar de un índice como `DataFrame.stack()`). Toma tres parámetros principales. 

* `id_vars`, que identifica cada observación
* `value_vars`, que aporta los nombre de las variables y sus valores
* `var_name` y `value_name`, que permite nombrar esas columnas nuevas

In [15]:
df_precios.reset_index(inplace=True)
df_precios.columns = ['Ciudad','Casa','Coche','Transporte']
df_precios

,Ciudad,Casa,Coche,Transporte
0,Madrid,10,40,1
1,Barcelona,20,50,2
2,Paris,30,60,3


In [16]:
df_precios.melt(id_vars='Ciudad', value_vars=['Casa','Coche'], var_name='Concepto', value_name='Precio')

,Ciudad,Concepto,Precio
0,Madrid,Casa,10
1,Barcelona,Casa,20
2,Paris,Casa,30
3,Madrid,Coche,40
4,Barcelona,Coche,50
5,Paris,Coche,60


<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>  La función`pd.wide_to_long()`  ofrece otra forma de pasar de ancho a largo cuando los nombres de las columnas siguen un patrón de prefijo y sufijo. Requiere una estructura de nombres más concreta que `melt()`, pero evita especificar manualmente cada columna en ciertos casos. [**(+ inf)**](https://pandas.pydata.org/docs/reference/api/pandas.wide_to_long.html).
</div>

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>